### Basic chatbot using graph API

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [ ]:
MODEL = "qwen/qwen3.6-27b"

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]  # add_messages is the reducer

graph_builder = StateGraph(State)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model=MODEL)

In [ ]:
llm

In [ ]:
# Node
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("llmchatbot", chatbot)

graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display, Markdown

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
# response = graph.invoke({"messages": "Hi"})

In [ ]:
# display(Markdown(response["messages"][-1].content))

In [ ]:
# for event in graph.stream({"messages":"Hi, my name is alex, please respond in plain text and in one line"}):
#     for values in event.values():
#         print(values["messages"][0].content)

### Chatbot with tools

In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)
# tool.invoke("what is langhraph")

In [ ]:
## Custom function
def multiply(a:int, b:int) -> int:
    """
    Multiply a and b
    Args:
        a (int): first int
        b (int): second int
    Returns
        int: a*b output int
    """
    return a*b

In [ ]:
tools = [tool, multiply]

In [ ]:
llm_with_tools = llm.bind_tools(tools)

In [ ]:
llm_with_tools

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

## Node def
def tool_calling_llm(state:State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Graph
graph_builder = StateGraph(State)
graph_builder.add_node("tool_calling_llm", tool_calling_llm)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "tool_calling_llm")
graph_builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition
)
graph_builder.add_edge("tools", END)

## Compile
graph = graph_builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# res = graph.invoke({"messages": "multiply 2 and 3"})
res = graph.invoke({"messages": "Get me the latest news of agentic AI"})

In [ ]:
res["messages"][-1].content

In [ ]:
res = graph.invoke({"messages": "What is 5 multiplied by 2 and then multiply by 8"})
res["messages"][-1].content

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

## Node def
def tool_calling_llm(state:State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Graph
graph_builder = StateGraph(State)
graph_builder.add_node("tool_calling_llm", tool_calling_llm)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "tool_calling_llm")
graph_builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition
)
graph_builder.add_edge("tools", "tool_calling_llm")

## Compile
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
res = graph.invoke({"messages": "Give me the latest AI news and then multiply 5 by 10"})
for m in res["messages"]:
    m.pretty_print()

### Adding memory in Agentic Graph

In [ ]:
res = graph.invoke({"messages": "Hello my name is alex"})
for m in res["messages"]:
    m.pretty_print()

In [ ]:
res = graph.invoke({"messages": "What is my name"})
for m in res["messages"]:
    m.pretty_print()

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

## Node def
def tool_calling_llm(state:State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Graph
graph_builder = StateGraph(State)
graph_builder.add_node("tool_calling_llm", tool_calling_llm)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "tool_calling_llm")
graph_builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition
)
graph_builder.add_edge("tools", "tool_calling_llm")

## Compile
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "1"}}

res = graph.invoke({"messages":"Hi my name is alex"}, config=config)

res["messages"][-1].content

In [ ]:
res = graph.invoke({"messages": "What is my name"}, config=config)
res["messages"][-1].content

### Streaming